# RT-DETR-L — відновлення `tune_reference` та повна COCO evaluation

Цей ноутбук **не тренує модель**. Він відновлює старий `epoch_04.pt` із Hugging Face (помилка `custom_criterion`), перевіряє отриманий checkpoint та оцінює **лише `tune_reference`** на всіх 2846 кадрах, використовуючи той самий протокол, що й попередній ноутбук: `step=1`, `tile=0`, `imgsz=640`, `conf=0.01`.

Перед запуском: **Internet ON**, **GPU T4**, competition dataset підключений, **Kaggle Secret `HF_TOKEN` дозволений для цього ноутбука**. Запускай усі комірки послідовно. Вихідний файл `tune_reference_recovered.pt` і фінальні метрики залишаються в Kaggle Output; виправлений checkpoint також збережемо в Hugging Face окремим файлом, не замінюючи оригінал.

**Примітка:** оригінальний `tune_reference/weights/best.pt` відсутній, тож оцінюється відновлена **остання (4-та) епоха**, а не гарантовано найкраща.

In [1]:
# 1. Install dependencies and define paths
import os, sys, subprocess, json, time
from pathlib import Path

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.210', 'pycocotools', 'huggingface_hub'])

import torch, ultralytics
from ultralytics import RTDETR
from huggingface_hub import HfApi, hf_hub_download
from kaggle_secrets import UserSecretsClient

assert ultralytics.__version__ == '8.4.210', ultralytics.__version__
assert torch.cuda.is_available(), 'Enable Kaggle GPU'

HF_REPO = 'lilialania/visdrone-retinanet'
DATA_ROOT = Path('/kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026')
REPO = Path('/kaggle/working/drone-multi-object-detection')
REPO_URL = 'https://github.com/lilialania/drone-multi-object-detection.git'
WORK = Path('/kaggle/working/reference_recovery_evaluation')
WORK.mkdir(parents=True, exist_ok=True)
VAL_ROOT = DATA_ROOT / 'val'
ANNOTATIONS = VAL_ROOT / 'annotations_coco.json'
assert ANNOTATIONS.is_file(), f'Competition data missing: {ANNOTATIONS}'

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN').strip()
assert HF_TOKEN, 'Kaggle HF_TOKEN missing'
os.environ['HF_TOKEN'] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
print('HF authenticated as:', api.whoami()['name'])
api.repo_info(repo_id=HF_REPO, repo_type='model')
print('HF repository access: OK')
print('GPU:', torch.cuda.get_device_name(0))

if not (REPO / 'src' / 'tiled_yolo.py').is_file():
    subprocess.check_call(['git','clone',REPO_URL,str(REPO)])
print('Repository:',REPO)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
HF authenticated as: lilialania
HF repository access: OK
GPU: Tesla T4


Cloning into '/kaggle/working/drone-multi-object-detection'...


Repository: /kaggle/working/drone-multi-object-detection


## 2. Відновлення checkpoint

Старий `.pt` містить прив'язаний метод `custom_criterion`. Тимчасово надаємо класу метод із такою назвою **тільки для десеріалізації**, після чого видаляємо старі прив'язані методи з об'єктів і зберігаємо новий `.pt`. Джерело checkpoint — власний приватний HF repository; **не використовуйте цей спосіб із неперевіреними `.pt` файлами**, оскільки `torch.load(..., weights_only=False)` може виконувати сторонній код.

In [2]:
# 2. Recover ONLY the legacy tune_reference checkpoint
from ultralytics.nn.tasks import RTDETRDetectionModel
from ultralytics.models.utils.loss import RTDETRDetectionLoss

legacy_remote = 'query_rtdetr/tune_reference/epoch_04.pt'
original = hf_hub_download(
    repo_id=HF_REPO, filename=legacy_remote, repo_type='model', token=HF_TOKEN
)
print('Legacy checkpoint:', legacy_remote)

# Old pickled bound method looks up `custom_criterion` on RTDETRDetectionModel.
def _legacy_custom_criterion(self):
    return RTDETRDetectionLoss(nc=self.model[-1].nc, use_vfl=True, aux_loss=True)

RTDETRDetectionModel.custom_criterion = _legacy_custom_criterion

# This is a trusted checkpoint from our own private repository.
checkpoint = torch.load(original, map_location='cpu', weights_only=False)
assert isinstance(checkpoint, dict), type(checkpoint)

for k in ('model', 'ema'):
    obj = checkpoint.get(k)
    if obj is None:
        continue
    # Remove serialized instance-level method overrides that caused the error.
    for attr in ('init_criterion', 'custom_criterion'):
        if attr in vars(obj):
            delattr(obj, attr)
    # Criterion is for training only and should be re-created, not serialized.
    if 'criterion' in vars(obj):
        delattr(obj, 'criterion')

recovered = WORK / 'tune_reference_recovered.pt'
torch.save(checkpoint, recovered)

# Remove temporary class patch before verifying clean deserialization.
del RTDETRDetectionModel.custom_criterion

check = RTDETR(str(recovered))
assert check.model.model[-1].nc == 8
print('RECOVERY VERIFIED: classes =', check.model.model[-1].nc)
print('RECOVERY VERIFIED: queries =', check.model.model[-1].num_queries)
print('Recovered file:', recovered, '| size MB:', round(recovered.stat().st_size/1024**2,2))
del check, checkpoint


query_rtdetr/tune_reference/epoch_04.pt: reconstructing file:   0%|          |  0.00B /  329MB            

query_rtdetr/tune_reference/epoch_04.pt: downloading bytes:           |  0.00B            

Legacy checkpoint: query_rtdetr/tune_reference/epoch_04.pt
RECOVERY VERIFIED: classes = 8
RECOVERY VERIFIED: queries = 300
Recovered file: /kaggle/working/reference_recovery_evaluation/tune_reference_recovered.pt | size MB: 314.14


In [3]:
# 3. Upload recovered checkpoint to Hugging Face without overwriting the original
recovered_remote = 'query_rtdetr/tune_reference/recovered_epoch_04.pt'
api.upload_file(
    path_or_fileobj=str(recovered),
    path_in_repo=recovered_remote,
    repo_id=HF_REPO,
    repo_type='model',
    commit_message='Recover RT-DETR reference epoch 04 for inference evaluation',
)
assert api.file_exists(repo_id=HF_REPO, filename=recovered_remote, repo_type='model')
print('BACKUP VERIFIED:', recovered_remote)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

BACKUP VERIFIED: query_rtdetr/tune_reference/recovered_epoch_04.pt


In [4]:
# 4. Write the SAME COCO evaluation runner as the previous notebook
# 10C. Запис окремого evaluation runner (в окремому процесі для кожної моделі)
# Runner використовує репозиторний sliced_predict і pycocotools; не тренує модель.
EVAL_SCRIPT = WORK / 'evaluate_one_rtdetr.py'
EVAL_SCRIPT.write_text(r'''import argparse, json, time
from pathlib import Path
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import RTDETR
from src.models.tiled_yolo import sliced_predict
from src.data_utils import image_path_for
from src.models.yolo_baseline import MAX_DETECTIONS

p=argparse.ArgumentParser()
p.add_argument('--weights', required=True)
p.add_argument('--split-root', required=True)
p.add_argument('--out', required=True)
p.add_argument('--step',type=int,default=1)
p.add_argument('--tile',type=int,default=0)
p.add_argument('--imgsz',type=int,default=640)
p.add_argument('--conf',type=float,default=.01)
a=p.parse_args()
coco=COCO(str(Path(a.split_root)/'annotations_coco.json'))
image_ids=sorted(coco.getImgIds())[::a.step]
model=RTDETR(a.weights)
results=[]
seconds=0.0
for i,image_id in enumerate(image_ids):
    tick=time.perf_counter()
    boxes,scores,classes=sliced_predict(
        model,image_path_for(Path(a.split_root),coco.imgs[image_id]['file_name']),
        a.tile,0.2,a.imgsz,a.conf)
    elapsed=time.perf_counter()-tick
    if i: seconds+=elapsed # first image is warm-up
    for (x1,y1,x2,y2),score,cls in zip(boxes,scores,classes):
        results.append({'image_id':int(image_id),'category_id':int(cls)+1,
                        'bbox':[float(x1),float(y1),float(x2-x1),float(y2-y1)],
                        'score':float(score)})
    if (i+1)%100==0 or i+1==len(image_ids):
        print(f'Predicted {i+1}/{len(image_ids)}',flush=True)
if not results: raise RuntimeError('No predictions produced; cannot evaluate')
ev=COCOeval(coco,coco.loadRes(results),'bbox')
ev.params.imgIds=image_ids
ev.params.maxDets=[1,100,MAX_DETECTIONS]
ev.evaluate();ev.accumulate();ev.summarize()
# Extract AP from maxDet=MAX_DETECTIONS using same precision tensor as src.models.faster_rcnn.coco_map
pr=ev.eval['precision'][...,-1] # [IoU, recall, class, area]
def ap(a):
    v=a[a>-1]
    return float(v.mean()) if len(v) else float('nan')
metrics={
    'mAP50_95':ap(pr[:,:,:,0]),
    'AP50':ap(pr[0,:,:,0]),
    'AP75':ap(pr[5,:,:,0]),
    'AP_small':ap(pr[:,:,:,1]),
    'AP_medium':ap(pr[:,:,:,2]),
    'AP_large':ap(pr[:,:,:,3]),
    'ms_per_image':(1000*seconds/(len(image_ids)-1)) if len(image_ids)>1 else float('nan'),
    'images_evaluated':len(image_ids),
    'detections':len(results),
    'step':a.step,'tile':a.tile,'imgsz':a.imgsz,'conf':a.conf,
    'max_detections':MAX_DETECTIONS,
}
Path(a.out).write_text(json.dumps(metrics,indent=2),encoding='utf-8')
print('METRICS_JSON:',json.dumps(metrics),flush=True)
''', encoding='utf-8')
print('Prepared:',EVAL_SCRIPT)

Prepared: /kaggle/working/reference_recovery_evaluation/evaluate_one_rtdetr.py


In [5]:
# 5. Evaluate ONLY tune_reference with the SAME full-validation protocol
STEP = 1
TILE = 0
IMGSZ = 640
CONF = 0.01

from pycocotools.coco import COCO
coco = COCO(str(ANNOTATIONS))
print('Validation images:',len(coco.getImgIds()))

metrics_file = WORK / 'tune_reference.json'
log_file = WORK / 'tune_reference.log'

cmd = [sys.executable, '-u', str(EVAL_SCRIPT),
       '--weights', str(recovered),
       '--split-root', str(VAL_ROOT),
       '--out', str(metrics_file),
       '--step', str(STEP),
       '--tile', str(TILE),
       '--imgsz', str(IMGSZ),
       '--conf', str(CONF)]

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO) + os.pathsep + env.get('PYTHONPATH', '')

if metrics_file.is_file():
    print('Existing evaluation found, no repeat:', metrics_file)
else:
    with log_file.open('w', encoding='utf-8') as logfile:
        result = subprocess.run(cmd, stdout=logfile, stderr=subprocess.STDOUT, env=env)
    print(log_file.read_text(encoding='utf-8',errors='replace')[-3500:])
    if result.returncode:
        raise RuntimeError(f'COCO evaluation failed; see {log_file}')

metrics = json.loads(metrics_file.read_text(encoding='utf-8'))
import pandas as pd
summary = pd.DataFrame([{'experiment':'tune_reference',**metrics}])
display(summary.T)
summary.to_csv(WORK/'tune_reference_coco_metrics.csv', index=False)
print('Evaluation saved:', metrics_file)


loading annotations into memory...
Done (t=0.70s)
creating index...
index created!
Validation images: 2846
loading annotations into memory...
Done (t=0.47s)
creating index...
index created!
Predicted 100/2846
Predicted 200/2846
Predicted 300/2846
Predicted 400/2846
Predicted 500/2846
Predicted 600/2846
Predicted 700/2846
Predicted 800/2846
Predicted 900/2846
Predicted 1000/2846
Predicted 1100/2846
Predicted 1200/2846
Predicted 1300/2846
Predicted 1400/2846
Predicted 1500/2846
Predicted 1600/2846
Predicted 1700/2846
Predicted 1800/2846
Predicted 1900/2846
Predicted 2000/2846
Predicted 2100/2846
Predicted 2200/2846
Predicted 2300/2846
Predicted 2400/2846
Predicted 2500/2846
Predicted 2600/2846
Predicted 2700/2846
Predicted 2800/2846
Predicted 2846/2846
Loading and preparing results...
DONE (t=1.67s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=135.54s).
Accumulating evaluation results...
DONE (t=9.93s).
 Average Precision  (AP) 

,0
experiment,tune_reference
mAP50_95,0.199696
AP50,0.382901
AP75,0.182164
AP_small,0.052457
AP_medium,0.186917
AP_large,0.350486
ms_per_image,99.869026
images_evaluated,2846
detections,684323


Evaluation saved: /kaggle/working/reference_recovery_evaluation/tune_reference.json


In [6]:
# 6. Pack all recovery and evaluation results into ZIP
from zipfile import ZipFile, ZIP_DEFLATED
bundle = Path('/kaggle/working/tune_reference_recovery_results.zip')
with ZipFile(bundle, 'w', ZIP_DEFLATED) as z:
    for path in WORK.rglob('*'):
        if path.is_file() and path != recovered:
            z.write(path, arcname=path.relative_to(WORK))
print('ZIP ready:', bundle)
print('Checkpoint preserved locally:', recovered)
print('Checkpoint preserved on Hugging Face:', recovered_remote)
print('No training and no evaluation of tune_queries200 or tune_matching.')


ZIP ready: /kaggle/working/tune_reference_recovery_results.zip
Checkpoint preserved locally: /kaggle/working/reference_recovery_evaluation/tune_reference_recovered.pt
Checkpoint preserved on Hugging Face: query_rtdetr/tune_reference/recovered_epoch_04.pt
No training and no evaluation of tune_queries200 or tune_matching.


## Після виконання

У Kaggle **Output** забери `tune_reference_recovery_results.zip`. Його можна додати до попереднього архіву `rtdetr_tuning_comparison.zip`: так матимеш результати повної COCO evaluation усіх трьох конфігурацій. **Ці checkpoints відрізняються способом вибору:** `tune_reference` — остання епоха, а інші два — `best.pt`, тому враховуй це у висновку.